# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library. You'll load the dataset, review its record sets, extract tabular data, conduct exploratory analysis, and visualize findings using entity `@id` references to ensure reproducibility and schema consistency.

### Dataset Source
- [FAIR² Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields within each record set. For clarity, all entities are referenced by their `@id` as required by the Croissant schema.

In [ ]:
# List all record sets and their associated field @ids
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<none>')}")
    if 'field' in rs:
        # 'field' might be a dict (single) or list (multiple)
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            fid = f['@id'] if isinstance(f, dict) else f  # sometimes field is just @id string
            print(f"    - {fid}")
    else:
        print("  Fields: None")
    print()

## 3. Data Extraction
Extract data from each record set using its `@id`. We load each record set into a DataFrame referenced by its `@id`.

_Choose the main tabular record set(s) based on the overview above._

In [ ]:
# Gather all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    # Use the `records` method to load records from each record set
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set: {rsid}")
    else:
        print(f"No records found for record set: {rsid}")

# For illustration, select the first DataFrame with data for further exploration
main_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_rs_id = rsid
        break
if main_rs_id:
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets with tabular data were found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering, normalization, and group-wise analysis.

**Fields and grouping will be referenced by their `@id`.**

_Below, select a numeric field and a group-by field using their `@id` (from the columns output above)._

In [ ]:
# You may need to adapt these @ids based on what you see above. For illustration, we guess common names.
# Replace these with the actual field @ids present in your DataFrame.
numeric_field_id = None
group_field_id = None

# Try to detect a likely numeric field from the first few columns by checking dtype
df = dataframes[main_rs_id]
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Choose the first numeric field
    print(f"Selected numeric field @id: {numeric_field_id}")
else:
    print("No numeric columns detected.")

# For group field, prefer a categorical field
cat_cols = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() < 10]
if cat_cols:
    group_field_id = cat_cols[0]
    print(f"Selected group field @id: {group_field_id}")
else:
    print("No suitable group-by columns detected.")

# Proceed if a numeric field is found
if numeric_field_id:
    # For the illustration, set a threshold at the 25th percentile
    threshold = df[numeric_field_id].dropna().quantile(0.25)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (25th percentile): {len(filtered_df)} records")
    # Normalize the numeric field
    norm_key = f"{numeric_field_id}_normalized"
    filtered_df[norm_key] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, norm_key]].head())
    # Group by group_field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        grouped_df.columns = [group_field_id, f"{numeric_field_id}_mean"]
        print(f"\nGrouped by {group_field_id} and computed mean of {numeric_field_id}:")
        display(grouped_df)
else:
    print("No numeric field identified for EDA.")

## 5. Visualization
Visualize the data distributions and relationships between fields using matplotlib. Fields are referenced by their `@id`.

_If available, visualize the normalized numeric field and/or group-wise averages._

In [ ]:
if numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    df[numeric_field_id].hist(ax=ax[0], bins=20)
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    ax[0].set_ylabel('Count')
    if group_field_id and group_field_id in df.columns:
        df.boxplot(column=numeric_field_id, by=group_field_id, ax=ax[1])
        ax[1].set_title(f"{numeric_field_id} by {group_field_id}")
        ax[1].set_xlabel(group_field_id)
        ax[1].set_ylabel(numeric_field_id)
        plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you loaded and explored the FAIR² dataset using `mlcroissant`, referencing all entities by their schema `@id`. You identified available record sets and fields, extracted records into a DataFrame, applied basic EDA such as filtering and grouping, and visualized distributions for numeric data fields. This workflow serves as a robust template for further, entity-consistent FAIR dataset analysis.